In [1]:
%cd ../
%load_ext autoreload
%autoreload 2


/home/christian/bachelor-project


In [2]:
# Persistence baseline model evaluation

import os
import pandas as pd
from loguru import logger
import json

from model.persistence.persistence_model import PersistenceModel


## Load measurements data


In [3]:
# Load the full measurements dataset
measurements_path = "data/measurements.parquet"
measurements_df = pd.read_parquet(measurements_path)
measurements_df = measurements_df.sort_values(by="record_date")
logger.info(f"Total measurements: {len(measurements_df)} rows")


2025-10-04 14:23:22.431 | INFO     | __main__:<module>:5 - Total measurements: 9542751 rows


## Split train/test


In [4]:
# Split into train and test (same split as other models)
end_date = pd.Timestamp('2025-04-01')
train_df = measurements_df[measurements_df['record_date'] <= end_date].copy()
test_df = measurements_df[measurements_df['record_date'] > end_date].copy()

logger.info(f"Train set: {len(train_df)} rows (up to {end_date})")
logger.info(f"Test set: {len(test_df)} rows (after {end_date})")

train_df.head()


2025-10-04 14:23:22.697 | INFO     | __main__:<module>:6 - Train set: 8766680 rows (up to 2025-04-01 00:00:00)
2025-10-04 14:23:22.698 | INFO     | __main__:<module>:7 - Test set: 776071 rows (after 2025-04-01 00:00:00)


,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
3832811,1031.0,0.4,-3.0,240.0,1.8,-1.0,4461801,0.0,0,3,2020-01-01,90.2,3158,0.0
6477667,-999.0,-999.0,-999.0,290.0,3.0,-999.0,8045843,-999.0,-999,3,2020-01-01,-999.0,6106,-999.0
4938593,1031.0,1.9,0.6,270.0,4.2,0.6,5962005,0.0,0,3,2020-01-01,91.1,4642,0.0
2307244,1029.9,2.1,0.1,270.0,1.5,1.0,2387258,0.0,0,3,2020-01-01,92.1,1605,0.0
5159681,1023.2,0.9,-0.4,250.0,4.7,0.6,6261133,0.0,0,3,2020-01-01,98.1,5109,0.0


## Initialize Persistence Model


In [5]:
# Initialize persistence model
# Using 72 history steps (12 hours) and 72 horizon steps (12 hours)
# to match the other models
model = PersistenceModel(
    history_steps=72,
    horizon_steps=72,
)
model


## Train the model (optional - does nothing)


In [6]:
# "Train" the model (optional - does nothing)
# The persistence model has no parameters to learn
# This is just here to match the ModelInterface
model.train(train_df)


2025-10-04 14:23:22.720 | INFO     | model.persistence.persistence_model:train:40 - Persistence model requires no training


## Save the model


In [7]:
# Save the model metadata
save_dir = "model/persistence"
os.makedirs(save_dir, exist_ok=True)
model.save(save_dir)
logger.info(f"Saved persistence model to {save_dir}")


2025-10-04 14:23:22.728 | INFO     | model.persistence.persistence_model:save:203 - Persistence model metadata saved to model/persistence
2025-10-04 14:23:22.729 | INFO     | __main__:<module>:5 - Saved persistence model to model/persistence


## Evaluate on test set


In [8]:
# Evaluate on test set
metrics = model.evaluate(test_df)


/home/christian/bachelor-project/model/persistence/persistence_model.py:262: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resample_group)
2025-10-04 14:25:36.256 | INFO     | model.persistence.persistence_model:evaluate:110 - Evaluating on 781156 sequences
2025-10-04 14:25:40.622 | INFO     | model.persistence.persistence_model:evaluate:178 - Persistence Model Evaluation: mae_u=1.4895, rmse_u=2.0337, mae_v=1.3683, rmse_v=1.8487, mae_speed=1.3031, rmse_speed=1.7556, mae_direction_deg=40.77°


## Display results


In [9]:
# Print detailed results
print("\\n=== Persistence Model Results ===")
print(f"MAE (u component): {metrics['mae_u']:.4f} m/s")
print(f"RMSE (u component): {metrics['rmse_u']:.4f} m/s")
print(f"MAE (v component): {metrics['mae_v']:.4f} m/s")
print(f"RMSE (v component): {metrics['rmse_v']:.4f} m/s")
print(f"MAE (wind speed): {metrics['mae_speed']:.4f} m/s")
print(f"RMSE (wind speed): {metrics['rmse_speed']:.4f} m/s")
print(f"MAE (wind direction): {metrics['mae_direction_deg']:.2f}°")

# Display metrics as DataFrame
pd.DataFrame([metrics])


\n=== Persistence Model Results ===
MAE (u component): 1.4895 m/s
RMSE (u component): 2.0337 m/s
MAE (v component): 1.3683 m/s
RMSE (v component): 1.8487 m/s
MAE (wind speed): 1.3031 m/s
RMSE (wind speed): 1.7556 m/s
MAE (wind direction): 40.77°


,mae_u,rmse_u,mae_v,rmse_v,mae_speed,rmse_speed,mae_direction_deg
0,1.489484,2.033746,1.368334,1.848668,1.303116,1.755568,40.772025


## Save metrics to JSON


In [10]:
# Save metrics to file
metrics_path = os.path.join(save_dir, "test_metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved to {metrics_path}")


2025-10-04 14:25:41.556 | INFO     | __main__:<module>:5 - Metrics saved to model/persistence/test_metrics.json
